In [1]:
!pip install plotly


[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: C:\Python311\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
try:
    df = pd.read_csv('ALL_UQ_PREDICTED.csv')
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
except FileNotFoundError:
    print("Error: 'ALL_UQ_PREDICTED.csv' not found. Please upload the file.")
    df = pd.DataFrame()

# ==========================================
# 2. EXTRACT FILTERS
# ==========================================
# Get all model columns (excluding 'actual' and bound columns)
# Bound columns end with _L or _U
model_cols = [c for c in df.columns if c != 'actual' and not c.endswith('_L') and not c.endswith('_U')]

# Parse metadata from column names: model_scheme_seed
models = set()
schemes = set()
seeds = set()

col_meta = {} # Map col_name -> (model, scheme, seed)

for col in model_cols:
    parts = col.split('_')
    if len(parts) >= 3:
        m, s, sd = parts[0], parts[1], parts[2]
        models.add(m)
        schemes.add(s)
        seeds.add(sd)
        col_meta[col] = (m, s, sd)

sorted_models = sorted(list(models))
sorted_schemes = sorted(list(schemes))
sorted_seeds = sorted(list(seeds), key=lambda x: int(x) if x.isdigit() else x)

# ==========================================
# 3. CREATE WIDGETS
# ==========================================
style = {'description_width': 'initial'}

w_model = widgets.Dropdown(options=['All'] + sorted_models, value='All', description='Model Arch:', style=style)
w_scheme = widgets.Dropdown(options=['All'] + sorted_schemes, value='All', description='Scheme:', style=style)
w_seed = widgets.Dropdown(options=['All'] + sorted_seeds, value='All', description='Seed:', style=style)

# ==========================================
# 4. PLOTLY FUNCTION
# ==========================================
def update_plotly_chart(model, scheme, seed):
    if df.empty:
        return

    # Identify columns to plot based on filters
    cols_to_plot = []
    for col in model_cols:
        m, s, sd = col_meta.get(col, (None, None, None))
        
        if model != 'All' and m != model: continue
        if scheme != 'All' and s != scheme: continue
        if seed != 'All' and sd != seed: continue
        
        cols_to_plot.append(col)
        
    # Initialize Plotly Figure
    fig = go.Figure()

    # 1. Add Actual Data (Always Black & Thicker)
    fig.add_trace(go.Scatter(
        x=df.index, 
        y=df['actual'],
        mode='lines',
        name='Actual',
        line=dict(color='black', width=3)
    ))

    # 2. Add Selected Models with Uncertainty Bounds
    # For each model, plot the point prediction and add a shaded confidence interval
    for col in cols_to_plot:
        col_lower = col + '_L'
        col_upper = col + '_U'
        
        # Check if bounds exist for this column
        has_bounds = col_lower in df.columns and col_upper in df.columns
        
        if has_bounds:
            # Add shaded region for uncertainty bounds
            fig.add_trace(go.Scatter(
                x=df.index,
                y=df[col_upper],
                fill=None,
                mode='lines',
                line_color='rgba(0,0,0,0)',
                showlegend=False,
                name=f'{col} Upper'
            ))
            
            fig.add_trace(go.Scatter(
                x=df.index,
                y=df[col_lower],
                fill='tonexty',
                mode='lines',
                line_color='rgba(0,0,0,0)',
                fillcolor='rgba(0,100,200,0.2)',
                name=f'{col} (95% CI)',
                showlegend=True
            ))
        
        # Add point prediction line
        fig.add_trace(go.Scatter(
            x=df.index, 
            y=df[col],
            mode='lines',
            name=col,
            line=dict(width=1.5),
            opacity=0.8,
            visible=True # Visible by default
        ))

    # Layout Updates
    fig.update_layout(
        title=f"Forecast Comparison with Uncertainty Quantification ({len(cols_to_plot)} models selected)",
        xaxis_title="Date",
        yaxis_title="Value",
        height=600, # Fixed height, scrollable legend
        hovermode="x unified", # Shows all values at the cursor point
        template="plotly_white",
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.01 # Place legend just outside the graph
        ),
        margin=dict(r=150) # Add margin right for the legend
    )
    
    # Add Range Slider at the bottom
    fig.update_xaxes(
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        )
    )

    fig.show()

# ==========================================
# 5. DISPLAY UI
# ==========================================
ui = widgets.HBox([w_model, w_scheme, w_seed])

out = widgets.interactive_output(update_plotly_chart, {
    'model': w_model, 
    'scheme': w_scheme, 
    'seed': w_seed
})

display(ui, out)

Output()